<a href="https://colab.research.google.com/github/Faalih408/Analisis_sentimen_menggunakan_SVM_-_Naive_Bayes/blob/main/Analisis_sentimen_menggunakan_SVM_%26_Naive_Bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Twitter Auth Token

In [ ]:
#@title Twitter Auth Token

twitter_auth_token = '943fee2674a1fa0b1586110a7'

In [ ]:
# Import required Python package
!pip install pandas

# Install Node.js (because tweet-harvest built using Node.js)
!sudo apt-get update
!sudo apt-get install -y ca-certificates curl gnupg #menginstal berbagai paket yang diperlukan untuk menginstal Node.js, seperti ca-certificates, curl, dan gnupg
!sudo mkdir -p /etc/apt/keyrings
!curl -fsSL https://deb.nodesource.com/gpgkey/nodesource-repo.gpg.key | sudo gpg --dearmor -o /etc/apt/keyrings/nodesource.gpg # Perintah ini mengunduh sebuah file kunci yang digunakan untuk memverifikasi keaslian paket Node.js yang akan diinstal

!NODE_MAJOR=20 && echo "deb [signed-by=/etc/apt/keyrings/nodesource.gpg] https://deb.nodesource.com/node_$NODE_MAJOR.x nodistro main" | sudo tee /etc/apt/sources.list.d/nodesource.list #Baris ini menambahkan informasi tentang lokasi di mana paket Node.js dapat ditemukan, sehingga sistem dapat mengunduh dan menginstalnya

!sudo apt-get update #Setelah sumber ditambahkan, daftar paket yang tersedia diperbarui, lalu Node.js versi 20 (sesuai dengan yang ditentukan di NODE_MAJOR) diinstal
!sudo apt-get install nodejs -y

!node -v #Perintah ini digunakan untuk memeriksa apakah Node.js telah terinstal dengan benar dan untuk melihat versi yang digunakan

In [ ]:
# Crawl Data

filename = 'Data_Penelitian_baru.csv'
search_keyword = 'covid-19'
limit = 1200

!npx --yes tweet-harvest@2.6.1 -o "{filename}" -s "{search_keyword}" -l {limit} --token {twitter_auth_token} # perintah untuk menjalankan paket Node.js secara langsung

In [ ]:
import pandas as pd

# Menentukan jalur ke file CSV Anda
file_path = f"tweets-data/{filename}"

# Membaca file CSV ke dalam DataFrame pandas
df = pd.read_csv(file_path, delimiter=",")

# Menampilkan DataFrame
display(df)

# Memeriksa panjang dari DataFrame
num_rows = len(df)
print(f"The number of rows in the DataFrame is: {num_rows}")

In [ ]:
num_tweets = len(df)
print(f"Jumlah tweet dalam dataframe adalah {num_tweets}.")

Preprocessing data

In [ ]:
import pandas as pd
import re
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Skripsi/Data_analisis_sentimen_fixx.csv", encoding='latin-1')
df.head()

In [ ]:
df = df[['full_text']]
df

In [ ]:
df.shape

In [ ]:
df = df.drop_duplicates()

case folding

In [ ]:
def case_folding(text):
  """Fungsi untuk mengubah semua huruf menjadi huruf kecil"""
  return text.lower()
df['text_case_folding'] = df['text_clean'].apply(case_folding)

Stopword Removal

In [ ]:
!pip install sastrawi

In [ ]:
# stopword
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

# Menambahkan kata 'yg' ke dalam daftar stopwords
stop = stopwords.words('indonesian')
stop.append('yg')


# Fungsi untuk menghapus stopwords dari teks
def remove_stopwords(text):
    if isinstance(text, str):  # Memastikan nilai adalah string sebelum memproses
        return ' '.join([word for word in text.split() if word not in stop]) # Membuat sebuah list yang hanya berisi kata-kata yang tidak ada dalam list
    else:
        return ''  # Mengembalikan string kosong jika nilai bukan string

# Menggunakan fungsi remove_stopwords pada kolom 'text_clean'
df['text_StopWord'] = df['text_case_folding'].apply(remove_stopwords)
df.head(50)


Tokenizing

In [ ]:
#Tokenizing
import nltk
# Download the necessary data for punkt tokenizer
nltk.download('punkt_tab')  # This line is added to download 'punkt_tab'
nltk.download('punkt') #download terhadap resource punkt agar teks dapat dipecah menjadi kalimat
from nltk.tokenize import sent_tokenize, word_tokenize
df['text_tokens'] = df['text_StopWord'].apply(lambda x: word_tokenize(x))
df.head()

Stemming

In [ ]:
#Stemming
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory


#import swifter
# create stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# stemmed
def stemmed_wrapper(term):
    return stemmer.stem(term)

term_dict = {} #untuk menyimpan kata-kata unik dan bentuk stem-nya
hitung=0

for document in df['text_tokens']:
    for term in document:
        if term not in term_dict:
            term_dict[term] = ' '
#Fungsi stemmed_wrapper digunakan untuk memanggil fungsi stemmer.stem() yang akan mengembalikan bentuk dasar dari kata tersebut.
print(len(term_dict))
print("------------------------")
for term in term_dict:
    term_dict[term] = stemmed_wrapper(term)
    hitung+=1
    print(hitung,":",term,":" ,term_dict[term])

print(term_dict)
print("------------------------")

# apply stemmed term to dataframe
def get_stemmed_term(document):
    return [term_dict[term] for term in document]

#script ini bisa dipisah dari eksekusinya setelah pembacaaan term selesai
df['text_steamindo'] = df['text_tokens'].apply(lambda x:' '.join(get_stemmed_term(x)))
df.head(30)

In [ ]:
import os

directory_path = '/content/drive/MyDrive/Skripsi'  # Remove the extra space at the end
if not os.path.exists(directory_path):
    os.makedirs(directory_path)

# Now save the file
df.to_csv(os.path.join(directory_path, 'Hasil_Perpocessing.csv'), index=False)

Proses Pelabelan

In [ ]:
import pandas as pd # Import the pandas library and alias it as 'pd'
data = pd.read_csv("/content/drive/MyDrive/Skripsi/Hasil_Perpocessing.csv")
data.head()

In [ ]:
!pip install preprocessor
!pip install textblob
!pip install wordcloud
!pip install nltk
!pip show transformers
!pip install transformers torch
!pip show torch

In [ ]:
import preprocessor as p
#from textblob import TextBlob
import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

nltk.download('punkt')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import pandas as pd



# Load tokenizer dan model
tokenizer = AutoTokenizer.from_pretrained("./model_indobertweet")
model = AutoModelForSequenceClassification.from_pretrained("./model_indobertweet")

# Buat pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# Misal data['text_steamindo'] berisi tweet
labels = []
for text in data['text_steamindo']:
    if isinstance(text, str):
        result = sentiment_pipeline(text)[0]
        label = result['label']
        if label == 'positive':
            labels.append('Positif')
        elif label == 'negative':
            labels.append('Negatif')
        else:
            labels.append('Netral')
    else:
        labels.append('None')

# Simpan ke DataFrame
data['label'] = labels

# Ringkasan hasil
print(data['label'].value_counts())


In [ ]:
import matplotlib.pyplot as plt

# Menghitung jumlah ulasan positif, negatif, dan netral
jumlah_positif = data[data['label'] == 'Positif'].shape[0]
jumlah_negatif = data[data['label'] == 'Negatif'].shape[0]
jumlah_netral = data[data['label'] == 'Netral'].shape[0]

# Membuat bagan
labels = ['Positif', 'Negatif', 'Netral']
sizes = [jumlah_positif, jumlah_negatif, jumlah_netral]
colors = ['lightgreen', 'maroon', 'lightblue']  # Tambahkan warna untuk netral
explode = (0.1, 0, 0)  # Sesuaikan efek pecah jika perlu

plt.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%', shadow=True, startangle=140)
plt.axis('equal')
plt.title('Hasil Pelabelan')

# Menampilkan bagan
plt.show()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Menggabungkan semua teks dengan sentimen Negatif menjadi satu string
text_positive = ' '.join(data[data['label'] == 'Negatif']['text_steamindo'])


# Membuat WordCloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_positive)

# Plot WordCloud
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('WordCloud Sentimen Negatif')
plt.show()

In [ ]:
import os

directory_path = '/content/drive/MyDrive/Skripsi'  # Remove the extra space at the end
if not os.path.exists(directory_path):
    os.makedirs(directory_path)

# Now save the file
data.to_csv(os.path.join(directory_path, 'Hasil_Pelabelan.csv'), index=False)

Proses TFIDF

In [ ]:
!pip install pandas scikit-learn

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from tabulate import tabulate

data = pd.read_csv("/content/drive/MyDrive/Skripsi/Hasil_Pelabelan.csv")
data.head()

In [ ]:
tfidf_vectorizer = TfidfVectorizer() #membuat Objek TF-IDF Vectorizer
tfidf_matrix = tfidf_vectorizer.fit_transform(data['text_steamindo']) #menghitung TF-IDF untuk setiap kata dalam setiap dokumen (tweet) dan menghasilkan matriks TF-IDF

#ekstrak TF dan IDF
terms = tfidf_vectorizer.get_feature_names_out()
idf = np.log(tfidf_matrix.shape[0]/(np.count_nonzero(tfidf_matrix.toarray(), axis=0) + 1))

#membuat dataframe TF-IDF
tfidf_df = pd.DataFrame({'term' : terms, 'idf' : idf})

#menambahkan TF (Term Frequency) ke dataframe
for i, doc in enumerate(data['text_steamindo']):
  tf = tfidf_matrix[i].toarray().flatten()
  tfidf_df[f'tf_{i}'] = tf

tfidf_df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Memisahkan data menjadi data latih dan data uji
X_train = data['text_steamindo']
# Inisialisasi vektorizer TF-IDF
tfidf_vectorizer = TfidfVectorizer()

# Fit vektorizer ke data
tfidf_vectorizer.fit(X_train)  # X_train adalah data teks yang digunakan untuk pembelajaran

# Proses vektorisasi teks
tfidf_matrix = tfidf_vectorizer.transform(X_train)

# Membuat WordCloud berdasarkan bobot TF-IDF dari hasil vektorisasi
wordcloud = WordCloud(width=800, height=400, background_color='white')
wordcloud.generate_from_frequencies(dict(zip(tfidf_vectorizer.get_feature_names_out(), tfidf_matrix.sum(axis=0).tolist()[0])))

# Menampilkan WordCloud
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')  # Menghilangkan sumbu x dan y
plt.title('WordCloud kata yang memiliki nilai bobot tertiggi')
plt.show()


Klasifikasi Naive Bayes

In [ ]:
import pandas as pd # Import the pandas library and alias it as 'pd'
data = pd.read_csv("/content/drive/MyDrive/Skripsi/Data_Skripsi_Jadi(edited).csv")
data.head()

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(data['text_hasil_prepro'], data['label'], test_size = 0.30, random_state = 42)

In [ ]:
#PEMBOBOTAN
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer() #membuat objek TF-IDF vectorizer
tfidf_train = tfidf_vectorizer.fit_transform(X_train) #langkah ini menghitung TF-IDF untuk setiap dokumen dalam data
tfidf_test = tfidf_vectorizer.transform(X_test) #menerapkan tfidf_vectorizer pada data testing X_test

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer #class yang digunakan untuk melakukan feature extraction pada teks

vectorizer = CountVectorizer() #melakukan feature extraction pada data teks
vectorizer.fit(X_train) #untuk melatih vectorizer dengan data x_train (data latih)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

#membuat model
nb = MultinomialNB()
nb.fit(tfidf_train, y_train) #untuk melatih model nb menggunakan data training tfidf_train dan label training y_train

In [ ]:
#dilakukan melatih dan menguji model machine learning
X_train = vectorizer.transform(X_train)
X_test = vectorizer.transform(X_test)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# menghasilkan prediksi menggunakan model yang telah dipasang
y_pred = nb.predict(X_test)  # mengganti X_test dengan data pengujian

# Hitung confusion matrix
matriks = confusion_matrix(y_test, y_pred)
matriks

In [ ]:
ax = plt.subplots()
ax = sns.heatmap(pd.DataFrame(matriks), annot=True, cmap="YlGnBu" ,fmt='g',
            annot_kws={"fontsize":13})

ax.set_xlabel('Predicted labels')
ax.set_ylabel('True labels')
ax.set_title('Confusion Matrix')
ax.xaxis.set_ticklabels(['Negative','Netral', 'Positive', ])
ax.yaxis.set_ticklabels(['Negative', 'Netral', 'Positive'])
plt.show()

In [ ]:
y_pred = nb.predict(tfidf_test)

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score

# Train Multinomial Naive Bayes classifier
clf = MultinomialNB()
clf.fit(X_train, y_train)

# memprediksi label untuk data pengujian
predicted = clf.predict(X_test)

# menghitung nilai accuracy
accuracy = accuracy_score(y_test, y_pred)

# Use 'weighted' average for multiclass classification
recall = recall_score(y_test, y_pred, average='weighted') # Changed average to 'weighted'
precision = precision_score(y_test, y_pred, average='weighted') # Changed average to 'weighted'
f1 = f1_score(y_test, y_pred, average='weighted') # Changed average to 'weighted'


print(f'confusion_matrix:\n {confusion_matrix(y_test, y_pred)}')
print("Accuracy: ", accuracy)
print("Recall: ", recall)
print("Precision: ", precision)
print("F1 Score: ", f1)
print(classification_report(y_test,y_pred))

Klasifikasi SVM kernel RBF

In [ ]:
import pandas as pd # Import the pandas library and alias it as 'pd'
data = pd.read_csv("/content/drive/MyDrive/Skripsi/Data_Skripsi_Jadi(edited).csv")
data.head()

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(data['text_hasil_prepro'], data['label'], test_size = 0.20, random_state = 42)

In [ ]:
#PEMBOBOTAN
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer() #membuat objek TF-IDF vectorizer
tfidf_train = tfidf_vectorizer.fit_transform(X_train) #langkah ini menghitung TF-IDF untuk setiap dokumen dalam data
tfidf_test = tfidf_vectorizer.transform(X_test) #menerapkan tfidf_vectorizer pada data testing X_test

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
from sklearn.svm import SVC
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer # Import TfidfVectorizer
from sklearn import metrics # Import metrics from scikit-learn
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score


# dengan asumsi tfidf_train dan tfidf_test sudah didefinisikan dari kode sebelumnya
clf = SVC(max_iter=-1, C=1200,kernel='rbf')
clf.fit(tfidf_train, y_train.values.ravel()) # fit model menggunakan data yang telah ditransformasi
y_pred = clf.predict(tfidf_test) # prediksi menggunakan data tes yang telah ditransformasi
print("clf score: ",clf.score(tfidf_test, y_test)) # mengevaluasi menggunakan data tes yang telah ditransformasi
print('====================================================\n')

# menghitung nilai accuracy
accuracy = accuracy_score(y_test, y_pred)
# Change average to 'weighted' for multiclass classification
recall = recall_score(y_test, y_pred, average='weighted', pos_label='Negatif')
precision = precision_score(y_test, y_pred, average='weighted', pos_label='Negatif')
f1 = f1_score(y_test, y_pred, average='weighted', pos_label='Negatif')


print(f'confusion_matrix:\n {confusion_matrix(y_test, y_pred)}')
print("Accuracy:", accuracy)
print("Recall:", recall)
print("Precision:", precision)
print("F1 Score:", f1)
print(classification_report(y_test,y_pred))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Hitung confusion matrix
matriks = confusion_matrix(y_test, y_pred)
matriks

In [ ]:
ax = plt.subplots()
ax = sns.heatmap(pd.DataFrame(matriks), annot=True, cmap="Reds" ,fmt='g',
            annot_kws={"fontsize":13})

ax.set_xlabel('Predicted labels')
ax.set_ylabel('True labels')
ax.set_title('Confusion Matrix')
ax.xaxis.set_ticklabels(['Negative','Netral', 'Positive', ])
ax.yaxis.set_ticklabels(['Negative', 'Netral', 'Positive'])
plt.show()

Klasifikasi SVM kernel Sigmoid

In [ ]:
import pandas as pd # Import the pandas library and alias it as 'pd'
data = pd.read_csv("/content/drive/MyDrive/Skripsi/Data_Skripsi_Jadi.csv")
data.head()

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(data['text_Hasil_prepro'], data['label'], test_size = 0.20, random_state = 42)

In [ ]:
#PEMBOBOTAN
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_train = tfidf_vectorizer.fit_transform(X_train)
tfidf_test = tfidf_vectorizer.transform(X_test)

In [ ]:
from sklearn.svm import SVC
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer # Import TfidfVectorizer
from sklearn import metrics # Import metrics from scikit-learn
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score


# Assuming tfidf_train and tfidf_test are already defined from your previous code
clf = SVC(max_iter=-1, C=1000,kernel='sigmoid')
clf.fit(tfidf_train, y_train.values.ravel()) # Fit the model using the transformed data
y_pred = clf.predict(tfidf_test) # Predict using the transformed test data
print("clf score: ",clf.score(tfidf_test, y_test)) # Evaluate using the transformed test data
print('====================================================\n')

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, average='binary', pos_label='Negatif')
precision = precision_score(y_test, y_pred, average='binary', pos_label='Negatif')
f1 = f1_score(y_test, y_pred, average='binary', pos_label='Negatif')

print(f'confusion_matrix:\n {confusion_matrix(y_test, y_pred)}')
print("Accuracy:", accuracy)
print("Recall:", recall)
print("Precision:", precision)
print("F1 Score:", f1)
print(classification_report(y_test,y_pred))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Hitung confusion matrix
matriks = confusion_matrix(y_test, y_pred)
matriks

In [ ]:
ax = plt.subplots()
ax = sns.heatmap(pd.DataFrame(matriks), annot=True, cmap="YlGnBu" ,fmt='g',
            annot_kws={"fontsize":13})

ax.set_xlabel('Predicted labels')
ax.set_ylabel('True labels')
ax.set_title('Confusion Matrix')
ax.xaxis.set_ticklabels(['Negative', 'Positive'])
ax.yaxis.set_ticklabels(['Negative', 'Positive'])
plt.show()